# 08 — Local MLflow evidence and promotion decisions

**Estimated time:** 40 minutes<br>
**Prerequisites:** 07 — Frozen regression evaluation<br>
**Learner-produced evidence:** a local run record and an adopt/reject/inconclusive assessment

## Learning objectives

- Inspect local experiment lineage without a tracking server.
- Require comparable complete reports before making a promotion decision.
- Apply absolute output gates in addition to relative macro-F1 improvement.

This notebook is a teaching interface over the reusable code in `src/`.
It uses only prepared local files. Run `make prepare-flight` before the trip;
no cell installs packages or downloads data.


## Why this matters

Metrics printed in a notebook are easy to lose, mislabel, or compare against the wrong data. Experiment tracking connects parameters, metrics, datasets, and artifacts into reviewable runs. Promotion then applies predeclared gates to comparable evidence. MLflow stores evidence; it does not decide whether the evidence is complete or whether the risk trade-off is acceptable.

## Key terms in plain language

- **experiment:** an organized collection of related runs answering a development question.
- **run:** one recorded execution with a defined method, inputs, parameters, metrics, tags, and artifacts.
- **parameter:** a configuration value such as prompt version or LoRA rank, usually treated as an input to the run.
- **metric:** a numeric measurement such as macro F1, validity rate, or p95 latency.
- **artifact:** a file attached to evidence, such as a manifest, error table, configuration, or adapter metadata.
- **lineage:** the traceable relationship among source data, processed data, code, configuration, model, and result.
- **promotion gate:** a rule declared before evaluation that maps complete evidence to an allowed decision.
- **adopt / reject / inconclusive:** use the change, decline it, or state that the available evidence cannot support either conclusion.


## Mental model — how to think about this

An experiment run is a lab notebook entry, not a trophy cabinet. A valid decision record joins four things: the baseline, the precisely named change, a comparable result, and the rule used to decide. Gates should work like compiled policy: the same evidence always produces the same decision, and missing evidence fails closed to `inconclusive`.

### Running example

The score for the LoRA answer is meaningful only beside its data fingerprint, base revision, adapter hash, prompt/configuration, evaluator version, and comparator. MLflow indexes that evidence. A predeclared gate then decides `adopt`, `reject`, or `inconclusive`; the tracking tool does not invent the decision.

### Questions to ask before continuing

- Can a reviewer trace every result to exact data, method, code, and configuration fingerprints?
- Were gates declared before the frozen result was visible?
- Is the comparator the strongest meaningful baseline on the same evaluation fingerprint?
- Which required artifact or slice would make the decision inconclusive if absent?


## Current best practices

**Guidance reviewed:** 2026-08-01. These are reasons to inspect future tool changes, not a claim that practice stops evolving.

- **Log inputs as lineage, not only filenames.** Record dataset source, schema/profile where appropriate, digest, split manifest, and evaluation fingerprint.
- **Keep run roles explicit.** Separate smoke, baseline, candidate, frozen evaluation, and remediation runs so partial evidence cannot masquerade as final evidence.
- **Predeclare multi-dimensional gates.** Require quality improvement while protecting schema validity, policy compliance, and operational limits; do not optimize one metric at all costs.
- **Separate practical significance from statistical uncertainty.** Declare a minimum useful gain, and for higher-stakes decisions estimate paired uncertainty on the same records plus training variance across seeds.
- **Store decision evidence as immutable artifacts.** Include the comparison table, gate configuration, failure reasons, and the selected decision vocabulary.
- **Use local tracking honestly.** A local SQLite-backed store is excellent offline evidence, but team access, durability, access control, and retention require a governed shared deployment later.

## Common mistakes and why they fail

- **Metric shopping.** Selecting whichever score improved after results defeats predeclared evaluation intent.
- **Moving thresholds after the run.** That converts a gate into post-hoc justification and requires a new experiment.
- **Treating MLflow as proof by itself.** Tracking preserves what was logged; it cannot verify omitted inputs or bad methodology.
- **Comparing smoke and full runs as peers.** Different record sets, methods, or fingerprints are intentionally incomparable.

### What kind of guidance is this?

A **specification** defines a technical contract; **tool guidance** describes current official library behavior; **risk guidance** is voluntary governance guidance; and a **course rule** is this project's deliberately conservative choice. Do not call all four a formal standard. The lesson is complete offline; these primary links are optional follow-up reading.

- **Tool guidance:** [MLflow experiment tracking documentation](https://mlflow.org/docs/latest/ml/tracking)
- **Tool guidance:** [MLflow dataset tracking and lineage documentation](https://mlflow.org/docs/latest/dataset/)


## Setup — run, do not edit

Run the next cell once. It verifies the dedicated local Python kernel, finds
this sample project, and enables supported offline flags **before** model or
tracking libraries are imported. A successful cell ends with `setup: ready`.

This is one defense layer, not proof that every native library is physically
incapable of networking. The flight-preparation manifest, cached assets,
socket-denial checks, and a Wi-Fi-off rehearsal provide the other layers.


In [ ]:
import sys
from importlib import import_module
from pathlib import Path

current = Path.cwd().resolve()
project_root = None
for candidate in (current, *current.parents):
    direct = candidate
    nested = candidate / "examples" / "local-finetuning"
    if (direct / "src" / "aai_local_finetuning").is_dir():
        project_root = direct
        break
    if (nested / "src" / "aai_local_finetuning").is_dir():
        project_root = nested
        break
if project_root is None:
    raise RuntimeError(
        "Cannot locate examples/local-finetuning. Open this notebook from the "
        "repository, or run `make notebook` from the repository root."
    )

expected_python = (project_root / ".venv" / "bin" / "python").resolve()
active_python = Path(sys.executable).resolve()
if not expected_python.is_file() or active_python != expected_python:
    raise RuntimeError(
        "Wrong notebook kernel. Run `make notebook` from the repository root, "
        "then select 'AAI Local Fine-Tuning (offline)'. "
        f"Active Python: {active_python}; expected: {expected_python}"
    )

source_root = str(project_root / "src")
if source_root not in sys.path:
    sys.path.insert(0, source_root)

enable_offline_environment = import_module(
    "aai_local_finetuning.offline"
).enable_offline_environment
enable_offline_environment()

{
    "setup": "ready",
    "kernel": "AAI Local Fine-Tuning (offline)",
    "python": str(active_python),
    "network_library_flags": "enabled",
    "note": "Continue to the lesson; this cell is setup, not an exercise.",
}

## Local tracking, not cloud tracking

MLflow uses a repository-local SQLite database and local artifact root.
Runs can record dataset fingerprints, model revision, adapter/config
hashes, predictions, reports, and measured metrics without credentials.


In [ ]:
import mlflow

from aai_local_finetuning.evaluation import (
    BaselineEvaluation,
    PromotionThresholds,
    decide_lora_promotion,
)
from aai_local_finetuning.learning import load_report, load_support_splits
from aai_local_finetuning.offline import verify_flight_manifest
from aai_local_finetuning.settings import (
    PROJECT_ROOT,
    load_settings,
    sha256_file,
)
from aai_local_finetuning.tracking import configure_local_mlflow
from aai_local_finetuning.training import (
    TrainingManifest,
    TrainingManifestError,
    capture_execution_contract,
    execution_contract_sha256,
    recheck_training_snapshot,
    require_valid_training_snapshot,
    shared_adapter_lock,
)

settings = load_settings()
verify_flight_manifest(settings)
configure_local_mlflow(settings)
runs = mlflow.search_runs(order_by=["start_time DESC"], max_results=20)
runs[
    [
        column
        for column in (
            "run_id",
            "tags.mlflow.runName",
            "tags.run_purpose",
            "metrics.intent/macro_f1",
            "metrics.output/json_schema_validity_rate",
        )
        if column in runs.columns
    ]
]

## Understand a run before writing one

An MLflow **experiment** groups related work. A **run** is one evidence
envelope: parameters describe inputs and configuration, metrics record
numeric results, tags make purpose searchable, and artifacts preserve
files such as reports and decisions. The local SQLite store is the index;
the artifact directory holds evidence files. This notebook writes one
decision run only after it has assembled the assessment below.


In [ ]:
tracking_contract = {
    "experiment": settings.tracking.experiment,
    "backend_store": settings.tracking.uri,
    "artifact_root": settings.tracking.artifact_root,
    "planned_run_purpose": "promotion_assessment",
    "required_lineage": [
        "evaluation fingerprint",
        "model revision",
        "adapter and configuration hashes",
        "training and evaluation source/runtime contracts",
        "locked thresholds",
        "reports and final decision",
    ],
}
tracking_contract

## Inventory full notebook reports

Promotion requires all meaningful baselines and the LoRA change on the
complete frozen set with identical evaluation fingerprints. The LoRA
report must also carry the same training-manifest fingerprint that was
validated before inference and is still current now. Every report must
carry one evaluation-time source/runtime hash, all of those hashes must
agree, and that contract must still match the live interpreter, platform,
governed source, and exact package set. Partial reports, missing methods,
or any lineage mismatch force `inconclusive`.


In [ ]:
splits = load_support_splits(settings)
report_dir = PROJECT_ROOT / "artifacts" / "notebook" / "evaluation"
required_methods = (
    "majority",
    "keyword-rule",
    "basic",
    "strong",
    "few_shot",
    "lora-change",
)
report_paths = {
    method: report_dir / f"full-{method}-report.json" for method in required_methods
}
report_status = {method: path.is_file() for method, path in report_paths.items()}
lineage_path = report_dir / "full-lora-change-training-manifest.json"
report_status

## Apply the decision contract

Defaults require schema validity ≥ 0.98, response-policy compliance ≥
0.95, unsupported-intent rate = 0, and an absolute macro-F1 gain ≥ 0.01
over the strongest meaningful baseline. These course defaults were shown
and locked before notebook 07 opened test. Majority is retained as a floor
but excluded from the meaningful-baseline competition. The point-estimate
gate is suitable for this lab, not a substitute for risk-based thresholds
and paired uncertainty in a higher-stakes decision.


In [ ]:
thresholds = PromotionThresholds()
thresholds.model_dump(mode="json")

## Persist the assessment with its contract

Now the notebook writes one MLflow run whose purpose is explicit. The
assessment artifact is useful whether the decision is adopt, reject, or
inconclusive. Re-running creates a new attempt with a new run ID instead
of silently replacing prior evidence. A shared adapter lock covers report
loading, lineage validation, the promotion calculation, and the MLflow
decision artifact commit, so one assessment cannot mix adapter generations.
The cell recaptures the execution contract immediately after logging; a
concurrent source or package change makes the run fail instead of leaving
a successful-looking decision envelope.


In [ ]:
lineage_matches = False
lineage_error = None
current_snapshot = None
current_manifest_sha256 = None
loaded_reports = {}
fingerprints = set()
counts = set()
evaluation_contract_hashes = set()
complete_and_comparable = False

with shared_adapter_lock(settings.adapter_dir):
    try:
        current_snapshot = require_valid_training_snapshot(
            settings.adapter_dir,
            config_path=(PROJECT_ROOT / "configs" / "training" / "lora.yaml"),
        )
        current_manifest = current_snapshot.manifest
        recorded_manifest = TrainingManifest.model_validate_json(
            lineage_path.read_text(encoding="utf-8")
        )
        current_manifest_sha256 = current_snapshot.manifest_sha256
        current_execution_contract = capture_execution_contract()
        current_execution_sha256 = execution_contract_sha256(current_execution_contract)
        lineage_matches = (
            recorded_manifest == current_manifest
            and sha256_file(lineage_path) == current_manifest_sha256
        )
        report_status["lora-training-lineage"] = lineage_matches
        if all(report_status.values()):
            loaded_reports = {
                method: load_report(path) for method, path in report_paths.items()
            }
            fingerprints = {
                report.evaluation_fingerprint for report in loaded_reports.values()
            }
            counts = {report.total_examples for report in loaded_reports.values()}
            evaluation_contract_hashes = {
                report.evaluation_execution_contract_sha256
                for report in loaded_reports.values()
            }
            complete_and_comparable = (
                len(fingerprints) == 1
                and counts == {len(splits.test)}
                and evaluation_contract_hashes == {current_execution_sha256}
                and loaded_reports["lora-change"].training_manifest_sha256
                == current_manifest_sha256
                and loaded_reports["lora-change"].training_execution_contract_sha256
                == current_manifest.execution_contract_sha256
            )
    except (OSError, ValueError, TrainingManifestError) as error:
        report_status["lora-training-lineage"] = False
        lineage_error = str(error)

    if complete_and_comparable:
        recheck_training_snapshot(current_snapshot)
        assessment = decide_lora_promotion(
            change_name="bitext-structured-output-lora-v1",
            training_snapshot=current_snapshot,
            change_report=loaded_reports["lora-change"],
            baselines=[
                BaselineEvaluation(
                    name=name,
                    report=loaded_reports[name],
                    meaningful=name != "majority",
                )
                for name in required_methods
                if name != "lora-change"
            ],
            thresholds=thresholds,
        ).model_dump(mode="json")
    else:
        assessment = {
            "decision": "inconclusive",
            "reasons": [
                "all six methods must be scored on the complete frozen set",
                "report counts and evaluation fingerprints must match",
                "the LoRA report must match the current success manifest",
                "all reports must match the current source/runtime contract",
            ],
            "available_reports": report_status,
            "lineage_error": lineage_error,
        }

    if current_snapshot is not None:
        recheck_training_snapshot(current_snapshot)
    decision_execution_contract = capture_execution_contract()
    decision_execution_sha256 = execution_contract_sha256(decision_execution_contract)
    if (
        complete_and_comparable
        and decision_execution_contract != current_execution_contract
    ):
        raise RuntimeError("source/runtime changed before decision tracking")
    with mlflow.start_run(run_name="notebook-promotion-assessment") as run:
        mlflow.set_tags(
            {
                "run_purpose": "promotion_assessment",
                "execution_mode": "offline_local",
                "decision": str(assessment["decision"]),
            }
        )
        mlflow.log_params(
            {
                f"threshold.{name}": value
                for name, value in thresholds.model_dump(mode="json").items()
            }
        )
        mlflow.log_dict(assessment, "decision/assessment.json")
        mlflow.log_param(
            "evaluation_execution_contract_sha256",
            decision_execution_sha256,
        )
        mlflow.log_dict(
            decision_execution_contract.model_dump(mode="json"),
            "runtime/evaluation-execution-contract.json",
        )
        if complete_and_comparable:
            mlflow.log_param(
                "evaluation_fingerprint",
                next(iter(fingerprints)),
            )
            mlflow.log_param(
                "training_manifest_sha256",
                current_manifest_sha256,
            )
            mlflow.log_param(
                "training_execution_contract_sha256",
                current_manifest.execution_contract_sha256,
            )
            mlflow.log_dict(
                current_manifest.execution_contract.model_dump(mode="json"),
                "change/training-execution-contract.json",
            )
            for name, value in loaded_reports["lora-change"].flat_metrics().items():
                mlflow.log_metric(f"change.{name}", value)
        if capture_execution_contract() != decision_execution_contract:
            raise RuntimeError(
                "source/runtime changed while decision evidence " "was being committed"
            )
        if current_snapshot is not None:
            recheck_training_snapshot(current_snapshot)
        decision_run_id = run.info.run_id
    if current_snapshot is not None:
        recheck_training_snapshot(current_snapshot)
{"decision_run_id": decision_run_id, "assessment": assessment}

## Exercise — defend the decision

Write a short rationale that cites the strongest meaningful baseline,
macro-F1 gain, schema gate, policy gate, and unsupported-intent gate—or
names the missing evidence that makes the decision inconclusive.


In [ ]:
decision_rationale = (
    "The current notebook run remains inconclusive unless all six full "
    "reports share the frozen fingerprint; partial metrics are not promotion evidence."
)
assert any(
    word in decision_rationale.lower() for word in ("adopt", "reject", "inconclusive")
)
decision_rationale

**Hint:** a decision is not a summary adjective. It is a reproducible
function over baseline, change, result, thresholds, and comparability.


## Checkpoint

You have completed the Bitext lifecycle without treating training as
success or using a partial run as promotion evidence.

**Next:** `09_capstone_policy_dataset.ipynb` asks a different question:
when should deterministic policy, not a language model, own the truth?
